<a href="https://colab.research.google.com/github/vcellmike/PatternsFormation/blob/main/Working/2024_08_21_XGBoost_on_ImageJ_Features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json
import os
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from sklearn.metrics import accuracy_score, classification_report
from transformers import AutoModel, AutoTokenizer, get_scheduler
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
from torch.optim import AdamW
from tqdm.notebook import tqdm, trange
from time import perf_counter
from PIL import Image
import pandas as pd
#from google.colab import drive
from PIL import Image
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import pickle

print("All dependencies present.")

/Users/mikhailblinov/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/mikhailblinov/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All dependencies present.


In [3]:
# set random seeds for repeatability
import numpy as np
import random

def set_seed(seed_val):
    random.seed(seed_val)
    np.random.seed(seed_val)
    torch.manual_seed(seed_val)
    torch.cuda.manual_seed_all(seed_val)

print("Seed set")

Seed set


In [4]:
seed_val = 42
set_seed(seed_val)

In [44]:
# TODO: Replace curr_df_np126.pkl with images_unclassified

with open("data/unclassified_features.pkl", 'rb') as f:
  feats_df = pickle.load(f) # deserialize using load()

feats_df


,Ua,Ui,Ga,Gi,Ba,Da,Di,pattern,noise,path,...,FeretAngle_inverted_mean,FeretAngle_inverted_std,MinFeret_inverted_mean,MinFeret_inverted_std,AR_inverted_mean,AR_inverted_std,Round_inverted_mean,Round_inverted_std,Solidity_inverted_mean,Solidity_inverted_std
0,0.011652,0.1066,0.115805,0.061599,-0.043244,0.007928,0.559698,0,2,0.png,...,135.000,0.0,200.0,0.0,1.000,0.0,1.000,0.0,1.000,0.0
1,0.025122,0.063822,0.071957,0.106327,-0.113131,0.009186,0.611871,1,2,1.png,...,45.000,0.0,200.0,0.0,1.001,0.0,0.999,0.0,0.979,0.0
2,0.014455,0.085124,0.091124,0.113974,-0.060333,0.001776,1.798215,0,2,2.png,...,135.000,0.0,200.0,0.0,1.000,0.0,1.000,0.0,1.000,0.0
3,0.034475,0.059956,0.080798,0.10037,-0.132577,0.008852,0.854084,1,2,3.png,...,135.000,0.0,200.0,0.0,1.000,0.0,1.000,0.0,0.981,0.0
4,0.025569,0.143322,0.154726,0.025138,-0.067635,0.009413,1.87632,0,2,4.png,...,135.000,0.0,200.0,0.0,1.000,0.0,1.000,0.0,1.000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39512,0.047659,0.179187,0.122823,0.048745,-0.080442,0.011393,0.468993,0,3,noise 3 21522.0.png,...,135.000,0.0,200.0,0.0,1.000,0.0,1.000,0.0,1.000,0.0
39513,0.049658,0.087822,0.124444,0.105756,-0.140723,0.008773,1.001359,1,7,noise 7 21167.0.png,...,135.288,0.0,200.0,0.0,1.001,0.0,0.999,0.0,0.920,0.0
39514,0.025269,0.096806,0.0973,0.124239,-0.046031,0.008247,0.752525,0,5,noise 5 21035.0.png,...,135.000,0.0,200.0,0.0,1.000,0.0,1.000,0.0,1.000,0.0
39515,0.029474,0.085933,0.103994,0.103636,-0.119675,0.008054,1.058641,1,3,noise 3 24575.0.png,...,135.000,0.0,200.0,0.0,1.001,0.0,0.999,0.0,0.795,0.0


In [20]:
print(feats_df.shape)
feats_df.head()

(39517, 111)


,Ua,Ui,Ga,Gi,Ba,Da,Di,pattern,noise,path,...,FeretAngle_inverted_mean,FeretAngle_inverted_std,MinFeret_inverted_mean,MinFeret_inverted_std,AR_inverted_mean,AR_inverted_std,Round_inverted_mean,Round_inverted_std,Solidity_inverted_mean,Solidity_inverted_std
0,0.011652,0.1066,0.115805,0.061599,-0.043244,0.007928,0.559698,0,2,0.png,...,135.0,0.0,200.0,0.0,1.000,0.0,1.000,0.0,1.000,0.0
1,0.025122,0.063822,0.071957,0.106327,-0.113131,0.009186,0.611871,1,2,1.png,...,45.0,0.0,200.0,0.0,1.001,0.0,0.999,0.0,0.979,0.0
2,0.014455,0.085124,0.091124,0.113974,-0.060333,0.001776,1.798215,0,2,2.png,...,135.0,0.0,200.0,0.0,1.000,0.0,1.000,0.0,1.000,0.0
3,0.034475,0.059956,0.080798,0.10037,-0.132577,0.008852,0.854084,1,2,3.png,...,135.0,0.0,200.0,0.0,1.000,0.0,1.000,0.0,0.981,0.0
4,0.025569,0.143322,0.154726,0.025138,-0.067635,0.009413,1.87632,0,2,4.png,...,135.0,0.0,200.0,0.0,1.000,0.0,1.000,0.0,1.000,0.0


In [21]:
print(feats_df.columns[13:111])

Index(['num_spots', 'Mean', 'Median', 'Area_mean', 'Area_std', 'X_mean',
       'X_std', 'Y_mean', 'Y_std', 'Perim._mean', 'Perim._std', 'BX_mean',
       'BX_std', 'BY_mean', 'BY_std', 'Width_mean', 'Width_std', 'Height_mean',
       'Height_std', 'Major_mean', 'Major_std', 'Minor_mean', 'Minor_std',
       'Angle_mean', 'Angle_std', 'Circ._mean', 'Circ._std', 'Feret_mean',
       'Feret_std', 'IntDen_mean', 'IntDen_std', '%Area_mean', '%Area_std',
       'RawIntDen_mean', 'RawIntDen_std', 'FeretX_mean', 'FeretX_std',
       'FeretY_mean', 'FeretY_std', 'FeretAngle_mean', 'FeretAngle_std',
       'MinFeret_mean', 'MinFeret_std', 'AR_mean', 'AR_std', 'Round_mean',
       'Round_std', 'Solidity_mean', 'Solidity_std', 'num_spots_inverted',
       'Mean_inverted', 'Median_inverted', 'Area_inverted_mean',
       'Area_inverted_std', 'X_inverted_mean', 'X_inverted_std',
       'Y_inverted_mean', 'Y_inverted_std', 'Perim._inverted_mean',
       'Perim._inverted_std', 'BX_inverted_mean', 'BX_

XGBOOST

In [22]:
#! pip install xgboost
#! pip install openpyxl
#! pip install tensorflow
#! pip install graphviz
#! pip install hyperopt

from sklearn.multioutput import MultiOutputRegressor
from sklearn.svm import SVR
import numpy as np
from sklearn.model_selection import RepeatedKFold
from numpy import absolute
from pandas import read_csv
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedKFold
from xgboost import XGBRegressor
import openpyxl
from xgboost import cv
from PIL import Image
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import operator
# for loading/processing the images
import tensorflow
from tensorflow.keras.utils import load_img
from tensorflow.keras.utils import img_to_array
from keras.applications.vgg16 import preprocess_input

# models
from keras.applications.vgg16 import VGG16
from keras.models import Model

# clustering and dimension reduction
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import pickle
from sklearn.ensemble import RandomForestRegressor
from sklearn import tree
import graphviz
from sklearn import metrics
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
from hyperopt import fmin, tpe, hp,STATUS_OK
from sklearn.model_selection import KFold, cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

print("All dependencies present")

All dependencies present


In [23]:
np.unique(feats_df["num_spots"].astype(float))

array([0.000e+00, 1.000e+00, 2.000e+00, 3.000e+00, 4.000e+00, 5.000e+00,
       6.000e+00, 7.000e+00, 8.000e+00, 9.000e+00, 1.000e+01, 1.100e+01,
       1.200e+01, 1.300e+01, 1.400e+01, 1.500e+01, 1.600e+01, 1.700e+01,
       1.800e+01, 1.900e+01, 2.000e+01, 2.100e+01, 2.200e+01, 2.300e+01,
       2.400e+01, 2.500e+01, 2.600e+01, 2.700e+01, 2.800e+01, 2.900e+01,
       3.000e+01, 3.100e+01, 3.200e+01, 3.300e+01, 3.400e+01, 3.500e+01,
       3.600e+01, 3.700e+01, 3.800e+01, 3.900e+01, 4.000e+01, 4.100e+01,
       4.200e+01, 4.300e+01, 4.400e+01, 4.500e+01, 4.600e+01, 4.700e+01,
       4.800e+01, 4.900e+01, 5.000e+01, 5.100e+01, 5.200e+01, 5.300e+01,
       5.400e+01, 5.500e+01, 5.600e+01, 5.700e+01, 5.800e+01, 5.900e+01,
       6.000e+01, 6.100e+01, 6.200e+01, 6.300e+01, 6.400e+01, 6.500e+01,
       6.600e+01, 6.700e+01, 6.800e+01, 6.900e+01, 7.000e+01, 7.100e+01,
       7.200e+01, 7.300e+01, 7.400e+01, 7.500e+01, 7.600e+01, 7.700e+01,
       7.800e+01, 7.900e+01, 8.000e+01, 8.100e+01, 

In [24]:
# scale data 
# uses Standard Scaler, range [-1, 1]

## X (independent variable) --> the feature values themselves
X = feats_df[feats_df.columns[13:111]].astype(float)

## Y (dependent variables) --> the parameter values of Negan and RTO themselves
y = feats_df[["Ua","Ui","Ga","Gi","Da","Di","Ba"]].astype(float)

#Scale data with standardscaler
scaling = StandardScaler()

# Use fit and transform method
scaling.fit(X)
X_scaled = scaling.transform(X)

# select 20 percent for testing
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42) #create training split



# df_train = feats_df.sample(frac = 0.80, random_state = 42)
# df_train.reset_index(inplace = True, drop = True)
# df_val_test = feats_df.drop(df_train.index)

# p_out = 1

# df_val_test = df_val_test.sample(frac = p_out, random_state = 42)
# df_val = df_val_test.sample(frac = 0.5, random_state = 42)
# df_test = df_val_test.drop(df_val.index)

# df_val.reset_index(inplace = True, drop = True)
# df_test.reset_index(inplace = True, drop = True)

In [25]:
print(X_test)
#print(y_train)

[[ 0.84427681 -0.56527056 -0.81564454 ... -0.30076773 -0.9992852
  -0.26485482]
 [ 1.04601638  0.03622808 -0.81564454 ... -0.30076773 -3.22997976
  -0.26485482]
 [-0.46172146  1.28776123  1.22602426 ... -0.30076773  0.56389092
  -0.26485482]
 ...
 [-0.47233933 -0.98652117 -0.81564454 ... -0.30076773  0.56389092
  -0.26485482]
 [-0.47233933 -0.98652117 -0.81564454 ... -0.30076773  0.56389092
  -0.26485482]
 [-0.42986785  0.56368322  1.22602426 ...  3.92583447 -0.52963874
   2.96037405]]


In [26]:
y_test

,Ua,Ui,Ga,Gi,Da,Di,Ba
27444,0.034815,0.086269,0.098214,0.111581,0.010052,1.299758,-0.097848
11068,0.030000,0.240000,0.075000,0.100000,0.013000,0.500000,-0.120000
21463,0.015112,0.146368,0.100549,0.063209,0.010806,0.530263,-0.178364
31013,0.027648,0.159852,0.166757,0.120432,0.011730,1.441355,-0.128601
16157,0.014156,0.136370,0.043016,0.117460,0.008501,0.260931,-0.058947
...,...,...,...,...,...,...,...
27390,0.050426,0.077152,0.085455,0.127332,0.011023,0.739167,-0.123279
3082,0.050221,0.165615,0.046623,0.175211,0.010444,1.380281,-0.058233
37277,0.034758,0.025755,0.161337,0.160499,0.011274,0.992567,-0.125318
31798,0.046418,0.174132,0.050813,0.173091,0.009446,0.788921,-0.105999


In [31]:
model = xgb.XGBRegressor(n_estimators=1000, max_depth=10, eta=0.1, subsample=0.7, colsample_bytree=0.8, num_boost_round=50, objective= "reg:squarederror", device = "cuda")
model.fit(X_train, y_train)

/Users/mikhailblinov/Library/Python/3.9/lib/python/site-packages/xgboost/core.py:158: UserWarning: [16:27:11] WARNING: /Users/runner/work/xgboost/xgboost/src/context.cc:196: XGBoost is not compiled with CUDA support.
  warnings.warn(smsg, UserWarning)
/Users/mikhailblinov/Library/Python/3.9/lib/python/site-packages/xgboost/core.py:158: UserWarning: [16:27:11] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "num_boost_round" } are not used.

  warnings.warn(smsg, UserWarning)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device='cuda', early_stopping_rounds=None,
             enable_categorical=False, eta=0.1, eval_metric=None,
             feature_types=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=10,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=1000,
             n_jobs=None, num_boost_round=50, ...)

In [39]:
## Save XGBoost Model to File
path = "data/XGB_model1.model"
model.save_model(path)

/Users/mikhailblinov/Library/Python/3.9/lib/python/site-packages/xgboost/core.py:158: UserWarning: [16:30:45] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)


In [41]:
loaded_model = xgb.XGBRegressor()

loaded_model.load_model(path)

#loaded_model.fit(X_train, y_train)

# make predictions
y_pred = loaded_model.predict(X_test)

pd.DataFrame(data=y_pred) 

# Columns - parameters 1-7
# Rows - Subsample of rows

print("Model loaded")

Model loaded


In [43]:
# GOAL: To determine the error rate in predicting the value of continuous, parameter data

# Converts test Y values to numpy array
true_vals = np.array(y_test)

# Outputs amount of true values
print(true_vals.shape)

# Outputs amount of predicted values
print(y_pred.shape)

# Sets counters for correct, incorrect, errors numpy array
correct = 0
incorrect = 0
p_errs = np.zeros(7)

# loops through each row in the true_vals array
for i in range(true_vals.shape[0]):
  # predicted value = predicted value from loop
  pred = y_pred[i]
  # true value = true value from loop
  true_val = true_vals[i]
  # Adds to error: absolute percent difference between true and predicted values
  p_errs += (np.abs((pred-true_val)/true_val))

# Outputs error percentages
print((p_errs/true_vals.shape[0])*100)

(7904, 7)
(7904, 7)
[146.50361786 120.20670433  87.90838201  72.57677275          inf
  53.51171384  56.57515089]


/var/folders/3p/pbgs38h94vdd8kjwbd33x40h0000gr/T/ipykernel_91517/1689613994.py:24: RuntimeWarning: divide by zero encountered in divide
  p_errs += (np.abs((pred-true_val)/true_val))


In [35]:
# make a dataframe with the names and feat_dfs and inverse_feat_dfs:
real_imj_df = {"path":[],"feats_df":[],"inverse_feats_df":[]}

reg_paths = os.listdir("./content/ImageJ_data/Inverse_resized/")

for path in reg_paths:
  if path != ".ipynb_checkpoints":
    real_imj_df["path"].append(path)
    real_imj_df["feats_df"].append(pd.read_csv("./content/ImageJ_data/Regular_resized/" + path))
    real_imj_df["inverse_feats_df"].append(pd.read_csv("./content/ImageJ_data/Inverse_resized/" + path))

real_df = pd.DataFrame(real_imj_df)

In [37]:
real_df

,path,feats_df,inverse_feats_df
0,bHLH2_OE_(green)_(0-141_thresh.tif.csv,Label A...,Label Are...
1,bHLH2_RNAi_(green)_(0-170).tif.csv,Label Area M...,Label Area M...
2,F2_260_green)_(0-105).tif.csv,Label Area M...,Label Area Mea...
3,F2_A 8_(green)_(0-100).tif.csv,Label Area M...,Label Area Me...
4,LF10_11-20-23_(green)_(0-152).tif.csv,Label ...,Label Area...
5,MLC_F1_11-20-23_(green)_(0-105).tif.csv,Label ...,Label Ar...
6,mpar_rto(c-c)_(green)_(0-137).tif.csv,Label Ar...,Label Area...
7,mpar_rto_crispr_(green)_(0-155).tif.csv,Label Ar...,Label Ar...
8,mpar_wt_(green)_(0-170).tif.csv,Label Area ...,Label Area M...
9,NEGAN_crispr_(green)_(0-150).tif.csv,Label Area ...,Label Area ...


In [38]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', None) 
real_df.shape

columns0 = ['feats_df', 'inverse_feats_df']

real_df2 = real_df.drop(columns=columns0)

real_df2

#real_df.head(14)

,path
0,bHLH2_OE_(green)_(0-141_thresh.tif.csv
1,bHLH2_RNAi_(green)_(0-170).tif.csv
2,F2_260_green)_(0-105).tif.csv
3,F2_A 8_(green)_(0-100).tif.csv
4,LF10_11-20-23_(green)_(0-152).tif.csv
5,MLC_F1_11-20-23_(green)_(0-105).tif.csv
6,mpar_rto(c-c)_(green)_(0-137).tif.csv
7,mpar_rto_crispr_(green)_(0-155).tif.csv
8,mpar_wt_(green)_(0-170).tif.csv
9,NEGAN_crispr_(green)_(0-150).tif.csv


In [39]:
# Add regular features

# first list is mean, second list is std except for mean and median
feats = {'Mean':[],'Median':[],'Area':[[],[]], 'X':[[],[]], 'Y':[[],[]], 'Perim.':[[],[]], 'BX':[[],[]],
         'BY':[[],[]], 'Width':[[],[]], 'Height':[[],[]], 'Major':[[],[]], 'Minor':[[],[]],
       'Angle':[[],[]], 'Circ.':[[],[]], 'Feret':[[],[]], 'IntDen':[[],[]], '%Area':[[],[]],
       'RawIntDen':[[],[]], 'FeretX':[[],[]], 'FeretY':[[],[]], 'FeretAngle':[[],[]], 'MinFeret':[[],[]], 'AR':[[],[]],
       'Round':[[],[]], 'Solidity':[[],[]]}

# feats = {'Mean_inverted':[],'Median_inverted':[],'Area':[[],[]], 'X':[[],[]], 'Y':[[],[]], 'Perim.':[[],[]], 'BX':[[],[]],
#          'BY':[[],[]], 'Width':[[],[]], 'Height':[[],[]], 'Major':[[],[]], 'Minor':[[],[]],
#        'Angle':[[],[]], 'Circ.':[[],[]], 'Feret':[[],[]], 'IntDen':[[],[]], '%Area':[[],[]],
#        'RawIntDen':[[],[]], 'FeretX':[[],[]], 'FeretY':[[],[]], 'FeretAngle':[[],[]], 'MinFeret':[[],[]], 'AR':[[],[]],
#        'Round':[[],[]], 'Solidity':[[],[]]}


counter = 0

# iterate through all sims
for n in range(real_df.shape[0]): # grab the feats dataframe for this sim
  fdf = real_df["feats_df"][n] #iterate through all feats within the feats dictionary

  for i in range(len(feats.keys())): # select the current feat
    feat = list(feats.keys())[i]

    # if feat in ['Mean_inverted','Median_inverted']: # check for feats that should be added from the overall measurement (first row)
    if feat in ['Mean','Median']: # check for feats that should be added from the overall measurement (first row)
      # feats[feat].append(fdf[feat[0:-9]][0]) #inverse
      feats[feat].append(fdf[feat[:]][0])
      # feats[feat].append(fdf[feat][0])

    else:
      if fdf.shape[0] <= 1: # check if the fdf is only one measurement (empty result)
        feats[feat][0].append(np.mean(np.array(fdf[feat][0]))) # mean of sole measurement
        feats[feat][1].append(np.std(np.array(fdf[feat][0]))) #std of sole measurement (0)

      else:
        feats[feat][0].append(np.mean(np.array(fdf[feat][1:]))) # add mean for each of the rest of the feats using the remaining rows
        feats[feat][1].append(np.std(np.array(fdf[feat][1:]))) # add std for each of the rest of the feats using the remaining rows

  if counter % 1000 == 0:
    print(counter)
  counter += 1

print("Counting done")

0
Counting done


In [40]:
num_spots = []

for i in range(real_df.shape[0]):
  fdf = real_df["feats_df"][i] #iterate through all feats within the feats dictionary
  if fdf.shape[0] <= 1:
    num_spots.append(0)
  else:
    num_spots.append(fdf.shape[0]-1)

# feats_df["num_spots_inverted"] = num_spots
real_df["num_spots"] = num_spots

In [41]:
for key in list(feats.keys()):
  # if key in ['Mean_inverted','Median_inverted']:
  if key in ['Mean','Median']:
    real_df[key] = feats[key]
  else:
    # feats_df[key + "_inverted_mean"] = feats[key][0]
    # feats_df[key + "_inverted_std"] = feats[key][1]
    real_df[key + "_mean"] = feats[key][0]
    real_df[key + "_std"] = feats[key][1]

In [42]:
# Add regular features

# first list is mean, second list is std except for mean and median
# feats = {'Mean':[],'Median':[],'Area':[[],[]], 'X':[[],[]], 'Y':[[],[]], 'Perim.':[[],[]], 'BX':[[],[]],
#          'BY':[[],[]], 'Width':[[],[]], 'Height':[[],[]], 'Major':[[],[]], 'Minor':[[],[]],
#        'Angle':[[],[]], 'Circ.':[[],[]], 'Feret':[[],[]], 'IntDen':[[],[]], '%Area':[[],[]],
#        'RawIntDen':[[],[]], 'FeretX':[[],[]], 'FeretY':[[],[]], 'FeretAngle':[[],[]], 'MinFeret':[[],[]], 'AR':[[],[]],
#        'Round':[[],[]], 'Solidity':[[],[]]}

feats = {'Mean_inverted':[],'Median_inverted':[],'Area':[[],[]], 'X':[[],[]], 'Y':[[],[]], 'Perim.':[[],[]], 'BX':[[],[]],
         'BY':[[],[]], 'Width':[[],[]], 'Height':[[],[]], 'Major':[[],[]], 'Minor':[[],[]],
       'Angle':[[],[]], 'Circ.':[[],[]], 'Feret':[[],[]], 'IntDen':[[],[]], '%Area':[[],[]],
       'RawIntDen':[[],[]], 'FeretX':[[],[]], 'FeretY':[[],[]], 'FeretAngle':[[],[]], 'MinFeret':[[],[]], 'AR':[[],[]],
       'Round':[[],[]], 'Solidity':[[],[]]}


counter = 0

# iterate through all sims
for n in range(real_df.shape[0]): # grab the feats dataframe for this sim
  fdf = real_df["inverse_feats_df"][n] #iterate through all feats within the feats dictionary

  for i in range(len(feats.keys())): # select the current feat
    feat = list(feats.keys())[i]

    if feat in ['Mean_inverted','Median_inverted']: # check for feats that should be added from the overall measurement (first row)
    # if feat in ['Mean','Median']: # check for feats that should be added from the overall measurement (first row)
      feats[feat].append(fdf[feat[0:-9]][0]) #inverse
      # feats[feat].append(fdf[feat[:]][0])

    else:
      if fdf.shape[0] <= 1: # check if the fdf is only one measurement (empty result)
        feats[feat][0].append(np.mean(np.array(fdf[feat][0]))) # mean of sole measurement
        feats[feat][1].append(np.std(np.array(fdf[feat][0]))) #std of sole measurement (0)

      else:
        feats[feat][0].append(np.mean(np.array(fdf[feat][1:]))) # add mean for each of the rest of the feats using the remaining rows
        feats[feat][1].append(np.std(np.array(fdf[feat][1:]))) # add std for each of the rest of the feats using the remaining rows

  if counter % 1000 == 0:
    print(counter)
  counter += 1

print("Counter complete")

0
Counter complete


In [43]:
num_spots = []

for i in range(real_df.shape[0]):
  fdf = real_df["inverse_feats_df"][i] #iterate through all feats within the feats dictionary
  if fdf.shape[0] <= 1:
    num_spots.append(0)
  else:
    num_spots.append(fdf.shape[0]-1)

real_df["num_spots_inverted"] = num_spots
# real_df["num_spots"] = num_spots

In [44]:
for key in list(feats.keys()):
  if key in ['Mean_inverted','Median_inverted']:
  # if key in ['Mean','Median']:
    real_df[key] = feats[key]
  else:
    real_df[key + "_inverted_mean"] = feats[key][0]
    real_df[key + "_inverted_std"] = feats[key][1]
    # real_df[key + "_mean"] = feats[key][0]
    # real_df[key + "_std"] = feats[key][1]

In [45]:
print(real_df.columns[3:101])

Index(['num_spots', 'Mean', 'Median', 'Area_mean', 'Area_std', 'X_mean', 'X_std', 'Y_mean', 'Y_std', 'Perim._mean', 'Perim._std', 'BX_mean', 'BX_std', 'BY_mean', 'BY_std', 'Width_mean', 'Width_std', 'Height_mean', 'Height_std', 'Major_mean', 'Major_std', 'Minor_mean', 'Minor_std', 'Angle_mean', 'Angle_std', 'Circ._mean', 'Circ._std', 'Feret_mean', 'Feret_std', 'IntDen_mean', 'IntDen_std', '%Area_mean', '%Area_std', 'RawIntDen_mean', 'RawIntDen_std', 'FeretX_mean', 'FeretX_std', 'FeretY_mean', 'FeretY_std', 'FeretAngle_mean', 'FeretAngle_std', 'MinFeret_mean', 'MinFeret_std', 'AR_mean', 'AR_std', 'Round_mean', 'Round_std', 'Solidity_mean', 'Solidity_std', 'num_spots_inverted', 'Mean_inverted', 'Median_inverted', 'Area_inverted_mean', 'Area_inverted_std', 'X_inverted_mean', 'X_inverted_std', 'Y_inverted_mean', 'Y_inverted_std', 'Perim._inverted_mean', 'Perim._inverted_std', 'BX_inverted_mean', 'BX_inverted_std', 'BY_inverted_mean', 'BY_inverted_std', 'Width_inverted_mean',
       'Width_

In [30]:
real_feats.head()
#real_df.head()

NameError: name 'real_feats' is not defined

In [52]:
new_df = pd.read_pickle("./data/Images_Classified_np126.pkl")

new_feats = new_df[new_df.columns[13:111]]

new_feats.head()

,num_spots,Mean,Median,Area_mean,Area_std,X_mean,X_std,Y_mean,Y_std,Perim._mean,...,FeretAngle_inverted_mean,FeretAngle_inverted_std,MinFeret_inverted_mean,MinFeret_inverted_std,AR_inverted_mean,AR_inverted_std,Round_inverted_mean,Round_inverted_std,Solidity_inverted_mean,Solidity_inverted_std
0,70,5.489,0,12.300000,2.548669,99.395943,58.506181,97.530229,58.913237,12.282143,...,45.0,0.0,200.0,0.0,1.001,0.0,0.999,0.0,0.979,0.0
1,63,4.921,0,12.253968,2.569468,101.800571,59.297600,99.451270,58.953924,11.663270,...,135.0,0.0,200.0,0.0,1.000,0.0,1.000,0.0,0.981,0.0
2,77,4.692,0,9.558442,1.427863,100.557675,58.022388,98.103338,59.633687,10.379961,...,45.0,0.0,200.0,0.0,1.000,0.0,1.000,0.0,0.982,0.0
3,73,7.931,0,17.041096,3.365572,103.192973,59.580318,99.320973,58.524761,13.973644,...,135.0,0.0,200.0,0.0,1.000,0.0,1.000,0.0,0.969,0.0
4,75,8.154,0,17.053333,3.905187,104.204653,61.052503,99.000587,59.296666,14.229160,...,135.0,0.0,200.0,0.0,1.000,0.0,1.000,0.0,0.968,0.0


In [29]:
real_df

NameError: name 'real_df' is not defined

In [64]:
new_df = pd.read_pickle("data/real_df_xgboost2.pkl")

new_feats = new_df[new_df.columns[3:111]]
print("AAA")
print(new_feats.head())


y_pred = None

loaded_model = xgb.XGBRegressor()

loaded_model.load_model('./content/Models/XGB_model1.model')

# # select features of real images
#real_feats = real_df[real_df.columns[13:111]]

#scale data
scaling=StandardScaler()

# Use fit and transform method
#scaling.fit(real_feats)
scaling.fit(new_feats)
new_feats_scaled = scaling.transform(new_feats)
print("BBB")
print(new_feats_scaled.shape)
#real_feats_scaled = scaling.transform(real_feats)

# predict params of images from real feats
y_pred = loaded_model.predict(new_feats_scaled)

print(y_pred[2])

new_df["pred_params"] = list(y_pred)

AAA
   num_spots     Mean  Median     Area_mean      Area_std      X_mean  \
0        101    5.699       0      8.851485     11.066304  125.673327   
1         49   23.479       0     75.163265    103.827240   99.147755   
2          2  251.322     255  19711.500000  19688.500000  148.632000   
3         77   15.306       0     31.181818     46.025890  101.898026   
4         32   62.124       0    304.531250    876.516507  104.642344   

       X_std      Y_mean      Y_std  Perim._mean  ...  \
0  57.972907   88.902515  52.059334    10.451525  ...   
1  55.940428   74.931612  51.325268    36.572388  ...   
2  49.216000   51.684500  49.271500   437.255000  ...   
3  40.868593  104.009818  57.481703    19.575195  ...   
4  49.916328   82.908906  45.964830    98.939469  ...   

   FeretAngle_inverted_mean  FeretAngle_inverted_std  MinFeret_inverted_mean  \
0                125.782500                 9.217500              100.500000   
1                135.000000                 0.000000  

In [46]:
new_df

,path,feats_df,inverse_feats_df,num_spots,Mean,Median,Area_mean,Area_std,X_mean,X_std,...,FeretAngle_inverted_mean,FeretAngle_inverted_std,MinFeret_inverted_mean,MinFeret_inverted_std,AR_inverted_mean,AR_inverted_std,Round_inverted_mean,Round_inverted_std,Solidity_inverted_mean,Solidity_inverted_std
0,LF10_11-20-23_(green)_(0-152).tif.csv,Label ...,Label Area...,101,5.699,0,8.851485,11.066304,125.673327,57.972907,...,125.782500,9.217500,100.500000,99.500000,1.505000,0.495000,0.745000,0.245000,0.989000,0.011000
1,mpar_rto(c-c)_(green)_(0-137).tif.csv,Label Ar...,Label Area...,49,23.479,0,75.163265,103.827240,99.147755,55.940428,...,135.000000,0.000000,200.000000,0.000000,1.010000,0.000000,0.990000,0.000000,0.908000,0.000000
2,mpar_rto_crispr_(green)_(0-155).tif.csv,Label Ar...,Label Ar...,2,251.322,255,19711.500000,19688.500000,148.632000,49.216000,...,85.368333,34.511534,7.389667,6.291349,2.470333,2.450747,0.641333,0.259052,0.912833,0.070862
3,MLC_F1_11-20-23_(green)_(0-105).tif.csv,Label ...,Label Ar...,77,15.306,0,31.181818,46.025890,101.898026,40.868593,...,135.000000,0.000000,200.000000,0.000000,1.006000,0.000000,0.994000,0.000000,0.940000,0.000000
4,RTO_rnai_med_(green)_(0-109).tif.csv,Label Are...,Label Area ...,32,62.124,0,304.531250,876.516507,104.642344,49.916328,...,136.734000,8.210170,34.848857,68.029623,1.249286,0.436381,0.865714,0.189952,0.895286,0.102845
5,NEGAN_crispr_(green)_(0-150).tif.csv,Label Area ...,Label Area ...,0,0.000,0,40000.000000,0.000000,100.000000,0.000000,...,135.000000,0.000000,200.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000
6,mpar_wt_(green)_(0-170).tif.csv,Label Area ...,Label Area M...,66,8.077,0,19.196970,18.371767,97.896258,52.566051,...,135.000000,0.000000,200.000000,0.000000,1.004000,0.000000,0.996000,0.000000,0.968000,0.000000
7,RTO_rnai_low_(green)_(0-115).tif.csv,Label Are...,Label Area ...,91,13.528,0,23.318681,39.296808,107.970220,46.996468,...,135.000000,0.000000,200.000000,0.000000,1.011000,0.000000,0.989000,0.000000,0.947000,0.000000
8,bHLH2_RNAi_(green)_(0-170).tif.csv,Label Area M...,Label Area M...,0,0.000,0,40000.000000,0.000000,100.000000,0.000000,...,135.000000,0.000000,200.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000
9,RTO_rnai_high_(green)_(0-103).tif.csv,Label Ar...,Label Ar...,11,128.246,255,1828.818182,5677.547083,113.048091,65.100510,...,117.005208,35.754028,9.485479,20.990371,1.680583,1.042379,0.696771,0.219777,0.882354,0.131232


In [65]:
new_df["pred_params"][0]

array([ 0.07444812,  0.1587155 ,  0.16584946,  0.14243883,  0.04621829,
        0.7816448 , -0.12434022], dtype=float32)

In [66]:
## TODO: Grab dataframe at the top (w/ all the features), drop all unnessesary columns (diff. from real_df, drop parameters)
## choose first five images and run it and see how close they are. 

for i in range(real_df.shape[0]):
  print(real_df["path"][i])
  print(real_df["pred_params"][i])

NameError: name 'real_df' is not defined